In [ ]:
#Bash no Azure
chmod +x criacaoGS.sh
sed -i 's/\r$//' criacaoGS.sh
./criacaoGS.sh

In [ ]:
#Conectando na máquina
ssh admin_fiap@<SEU_IP_PUBLICO>

In [ ]:
# 1. Clonar o projeto
git clone https://github.com/EnzoVazz/StellarGear.API.git
cd StellarGear.API

# 2. Configurar a Connection String para a rede interna do Docker
cat <<EOF > StellarGear.API/appsettings.json
{
  "Logging": {
    "LogLevel": {
      "Default": "Information",
      "Microsoft.AspNetCore": "Warning"
    }
  },
  "AllowedHosts": "*",
  "ConnectionStrings": {
    "OracleConnection": "Data Source=db_RM_561722:1521/XEPDB1;User Id=system;Password=Fiap_devops_SG;"
  }
}
EOF

# 3. Rodar as Migrations do EF Core na VM
docker run --rm -it \
  -v $(pwd):/src \
  --network stellargear-network \
  -w /src \
  mcr.microsoft.com/dotnet/sdk:10.0 \
  bash -c "dotnet restore && dotnet tool install --global dotnet-ef && export PATH=\"\$PATH:/root/.dotnet/tools\" && dotnet ef database update --project StellarGear.Infrastructure --startup-project StellarGear.API"

In [ ]:
#Acessando o sqlplus
docker exec -it db_RM_561722 sqlplus system/"Fiap_devops_SG"@//localhost:1521/XEPDB1

In [ ]:
#Populando o Banco com dados
DECLARE
    v_id_pass NUMBER;
    v_id_med NUMBER;
    v_id_traje NUMBER;
BEGIN
    INSERT INTO "medico" ("nome", "crm", "especialidade", "DataCriacao")
    VALUES ('Dr. Hans', 'CRM-1001', 'Medicina Aeroespacial', SYSDATE)
    RETURNING "id_medico" INTO v_id_med;

    INSERT INTO "passageiro" ("nome", "cpf", "idade", "status_medico", "DataCriacao")
    VALUES ('Aron Turista', '111.111.111-11', 34, 'APTO', SYSDATE)
    RETURNING "id_passageiro" INTO v_id_pass;

    INSERT INTO "traje" ("id_passageiro", "codigo_rfid", "dt_alocacao", "DataCriacao")
    VALUES (v_id_pass, 'RFID-X1A', SYSDATE, SYSDATE)
    RETURNING "id_traje" INTO v_id_traje;

    FOR i IN 1..85 LOOP
        INSERT INTO "leitura_sensor" ("id_traje", "temperatura", "humidade", "batimentos", "dt_leitura", "DataCriacao")
        VALUES (v_id_traje, 36.5 + MOD(i, 5), 50.0 + MOD(i, 10), 80 + MOD(i, 20), SYSDATE - (i/24), SYSDATE);
    END LOOP;
    
    COMMIT;
END;
/
EXIT;

In [ ]:
#Entrando na raiz
cd ~/StellarGear.API

# Criando o Dockerfile
cat <<EOF > Dockerfile
FROM mcr.microsoft.com/dotnet/sdk:10.0 AS build
WORKDIR /src
COPY . .
WORKDIR "/src/StellarGear.API"
RUN dotnet restore "StellarGear.API.csproj"
RUN dotnet build "StellarGear.API.csproj" -c Release -o /app/build
RUN dotnet publish "StellarGear.API.csproj" -c Release -o /app/publish

FROM mcr.microsoft.com/dotnet/aspnet:10.0
WORKDIR /app
RUN useradd -m fiapuser
USER fiapuser

COPY --from=build /app/publish .

EXPOSE 8080
ENV ASPNETCORE_URLS=http://+:8080
ENV ASPNETCORE_ENVIRONMENT=Production

ENTRYPOINT ["dotnet", "StellarGear.API.dll"]
EOF

# Fazendo o Build e subindo a API
docker build -t stellargear-api-image .
docker run -d --name api_RM_561722 -p 8080:8080 --network stellargear-network stellargear-api-image